# CNN-TCN Model for Ultrasound-to-Angle Prediction

## 🎯 Goal
Predict finger joint angles (MCP, PIP) from wrist ultrasound images using CNN-TCN hybrid model.

**Model**: ResNet-18 (CNN) + Temporal Convolutional Network (TCN)

**Loss**: MAE (Mean Absolute Error)

---

### ⚠️ Before Running:
1. **Enable GPU**: Runtime → Change runtime type → GPU (T4)
2. **Upload your data** to Google Drive or directly to Colab

## 1. Setup and Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages (most are pre-installed in Colab)
!pip install -q torch torchvision tqdm tensorboard

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (Optional)

If you store data in Google Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# After mounting, your data path will be:
# /content/drive/MyDrive/your_folder_name/

## 3. Upload Data

### Option A: Upload ZIP file
Upload your data as a ZIP file and extract it:

In [ ]:
# Upload the data.zip file (containing images and CSV files)
from google.colab import files
import zipfile
import os

print("Please upload your data.zip file...")
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content/data')
        print(f"✓ Extracted {filename}")

!ls -la /content/data

### Option B: Use data from Google Drive

In [ ]:
# If data is already in Google Drive, set the path:
DATA_DIR = '/content/drive/MyDrive/ultrasound_data'  # Modify this path
IMAGE_DIR = '/content/drive/MyDrive/ultrasound_data/251002_wrist'

# Check if data exists
!ls -la {DATA_DIR}

## 4. Define Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class TemporalConvNet(nn.Module):
    """TCN with Dilated Causal Convolutions"""
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)

        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            padding = (kernel_size - 1) * dilation_size

            layers += [
                nn.Conv1d(in_channels, out_channels, kernel_size,
                         padding=padding, dilation=dilation_size),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.BatchNorm1d(out_channels)
            ]

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class CNNBackbone(nn.Module):
    """ResNet-18 for spatial features"""
    def __init__(self, feature_dim=512):
        super(CNNBackbone, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = feature_dim

    def forward(self, x):
        features = self.features(x)
        features = features.view(features.size(0), -1)
        return features


class CNN_TCN_Model(nn.Module):
    """Complete CNN-TCN Model"""
    def __init__(self, window_size=7, tcn_channels=[256, 128, 64], dropout=0.2):
        super(CNN_TCN_Model, self).__init__()
        self.window_size = window_size
        
        # CNN for spatial features
        self.cnn = CNNBackbone(feature_dim=512)
        
        # TCN for temporal modeling
        self.tcn = TemporalConvNet(
            num_inputs=512,
            num_channels=tcn_channels,
            kernel_size=3,
            dropout=dropout
        )
        
        # Regression head
        final_tcn_dim = tcn_channels[-1]
        self.regression_head = nn.Sequential(
            nn.Linear(final_tcn_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)  # [MCP, PIP]
        )

    def forward(self, x):
        batch_size, window_size, C, H, W = x.shape
        
        # Extract CNN features
        x = x.view(batch_size * window_size, C, H, W)
        cnn_features = self.cnn(x)
        cnn_features = cnn_features.view(batch_size, window_size, -1)
        
        # TCN for temporal modeling
        cnn_features = cnn_features.transpose(1, 2)
        tcn_features = self.tcn(cnn_features)
        tcn_output = tcn_features[:, :, -1]
        
        # Regression
        predictions = self.regression_head(tcn_output)
        return predictions

print("✓ Model architecture defined")

## 5. Define Dataset and DataLoader

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class UltrasoundAngleDataset(Dataset):
    def __init__(self, csv_file, image_dir, window_size=7, transform=None):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.window_size = window_size
        self.transform = transform
        self.valid_indices = list(range(window_size - 1, len(self.data)))

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        start_idx = actual_idx - self.window_size + 1
        end_idx = actual_idx + 1

        # Load images
        images = []
        for i in range(start_idx, end_idx):
            img_filename = self.data.iloc[i]['image_filename']
            img_path = os.path.join(self.image_dir, img_filename)
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            images.append(image)

        images = torch.stack(images, dim=0)
        
        # Target angles
        mcp_angle = float(self.data.iloc[actual_idx]['MCP_Flex'])
        pip_angle = float(self.data.iloc[actual_idx]['PIP_Flex'])
        target = torch.tensor([mcp_angle, pip_angle], dtype=torch.float32)

        return images, target


def get_transforms(train=True, img_size=224):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.RandomRotation(degrees=5),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

print("✓ Dataset and transforms defined")

## 6. Configuration and Setup

In [ ]:
# ===== CONFIGURATION =====
# Modify these paths according to where you uploaded your data
DATA_DIR = '/content/data'  # Directory containing train.csv, val.csv, test.csv
IMAGE_DIR = '/content/data/251002_wrist'  # Directory containing ultrasound images

# Model hyperparameters
CONFIG = {
    'window_size': 7,
    'batch_size': 16,
    'epochs': 50,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'num_workers': 2,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create dataloaders
train_dataset = UltrasoundAngleDataset(
    os.path.join(DATA_DIR, 'train.csv'),
    IMAGE_DIR,
    window_size=CONFIG['window_size'],
    transform=get_transforms(train=True)
)

val_dataset = UltrasoundAngleDataset(
    os.path.join(DATA_DIR, 'val.csv'),
    IMAGE_DIR,
    window_size=CONFIG['window_size'],
    transform=get_transforms(train=False)
)

test_dataset = UltrasoundAngleDataset(
    os.path.join(DATA_DIR, 'test.csv'),
    IMAGE_DIR,
    window_size=CONFIG['window_size'],
    transform=get_transforms(train=False)
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], 
                         shuffle=True, num_workers=CONFIG['num_workers'])
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], 
                       shuffle=False, num_workers=CONFIG['num_workers'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], 
                        shuffle=False, num_workers=CONFIG['num_workers'])

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train batches: {len(train_loader)}")

## 7. Initialize Model and Training Components

In [ ]:
# Create model
model = CNN_TCN_Model(
    window_size=CONFIG['window_size'],
    tcn_channels=[256, 128, 64],
    dropout=0.2
).to(device)

# Loss and optimizer
criterion = nn.L1Loss()  # MAE
optimizer = torch.optim.Adam(model.parameters(), 
                            lr=CONFIG['learning_rate'],
                            weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("✓ Model initialized")

## 8. Training Loop

In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_mae': [],
    'mcp_mae': [],
    'pip_mae': []
}

best_val_loss = float('inf')

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(loader, desc='Training')
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        predictions = model(images)
        loss = criterion(predictions, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return running_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, targets in pbar:
            images, targets = images.to(device), targets.to(device)
            predictions = model(images)
            loss = criterion(predictions, targets)
            
            running_loss += loss.item()
            all_preds.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    
    avg_loss = running_loss / len(loader)
    mae = np.mean(np.abs(all_preds - all_targets))
    mcp_mae = np.mean(np.abs(all_preds[:, 0] - all_targets[:, 0]))
    pip_mae = np.mean(np.abs(all_preds[:, 1] - all_targets[:, 1]))
    
    return avg_loss, mae, mcp_mae, pip_mae


print("Starting training...\n")
print("=" * 70)

for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
    print("-" * 70)
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_mae, mcp_mae, pip_mae = validate(model, val_loader, criterion, device)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)
    history['mcp_mae'].append(mcp_mae)
    history['pip_mae'].append(pip_mae)
    
    # Print metrics
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val MAE: {val_mae:.4f}°")
    print(f"MCP MAE: {mcp_mae:.4f}°, PIP MAE: {pip_mae:.4f}°")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, 'best_model.pth')
        print(f"✓ Best model saved! (Val Loss: {val_loss:.4f})")

print("\n" + "=" * 70)
print("Training completed!")
print("=" * 70)

## 9. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MAE)')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# MAE plot
axes[1].plot(history['mcp_mae'], label='MCP MAE')
axes[1].plot(history['pip_mae'], label='PIP MAE')
axes[1].plot(history['val_mae'], label='Overall MAE', linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (degrees)')
axes[1].set_title('Validation MAE per Joint')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best validation loss: {best_val_loss:.4f}")

## 10. Test Set Evaluation

In [ ]:
# Load best model
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Evaluate on test set
test_loss, test_mae, test_mcp_mae, test_pip_mae = validate(model, test_loader, criterion, device)

print("\n" + "=" * 70)
print("TEST SET RESULTS")
print("=" * 70)
print(f"Test Loss: {test_loss:.4f}")
print(f"Overall MAE: {test_mae:.4f}°")
print(f"MCP MAE: {test_mcp_mae:.4f}°")
print(f"PIP MAE: {test_pip_mae:.4f}°")
print("=" * 70)

## 11. Download Trained Model

In [ ]:
# Download the trained model
from google.colab import files

files.download('best_model.pth')
files.download('training_history.png')

print("✓ Model and training history downloaded!")

## 12. Prediction Example

In [ ]:
# Test on a few samples
model.eval()
sample_images, sample_targets = next(iter(test_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    predictions = model(sample_images)

predictions = predictions.cpu().numpy()
sample_targets = sample_targets.numpy()

print("Sample Predictions vs Ground Truth:")
print("=" * 70)
print(f"{'Sample':<10} {'Pred MCP':<12} {'True MCP':<12} {'Pred PIP':<12} {'True PIP':<12} {'Error':<10}")
print("-" * 70)

for i in range(min(10, len(predictions))):
    pred_mcp, pred_pip = predictions[i]
    true_mcp, true_pip = sample_targets[i]
    error = np.abs(predictions[i] - sample_targets[i]).mean()
    print(f"{i+1:<10} {pred_mcp:<12.2f} {true_mcp:<12.2f} {pred_pip:<12.2f} {true_pip:<12.2f} {error:<10.2f}°")

print("=" * 70)